In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os

# Add the parent directory to Python's path
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import os
import src.model_architectures
from src.model_architectures import run_power_lstm, run_vanilla_transformer, run_univariate_patchtst, run_keras_patchtst, run_chronos_bolt, run_chronos2_multivariate, run_chronos_foundation
from src.ml_baselines import xgboost_implementation, rf_implementation
from src.evaluation_utils import save_experiment_results, plot_predictions, calculate_metrics
import importlib
import src.evaluation_utils
importlib.reload(src.evaluation_utils)
importlib.reload(src.model_architectures)
import importlib
import matplotlib.pyplot as plt
import random
import torch
import tensorflow as tf
import warnings
import logging

In [ ]:
#######################  60 - Minute Horizon Baselines #######################

# =======================================================
# 1. GLOBAL CONFIGURATION (24 steps/day)
# =======================================================
HORIZON = "60min"
RANDOM_SEED = 42
horizon_metrics = []

SEQ_LENGTH = 24       # 24-hour sequence length for Day-Ahead
MASE_M = 24           # Seasonal denominator for MASE metric (24 hours)
PLOT_WINDOW = 168     # 7-day visual window for plots (24 * 7)

# UNIVERSAL SEED LOCK
def set_global_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    print(f"Global seed locked to {seed} for reproducibility.")

set_global_seed(RANDOM_SEED)

# =======================================================
# 2. LOAD & SPLIT DATA
# =======================================================
df = pd.read_csv('../data/processed/nanogrid_60min_features.csv', index_col=0, parse_dates=True)
TARGET_COL = 'energy_consumption'

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Deterministic Split (70/10/20)
total_rows = len(df)
train_idx = int(total_rows * 0.70)
val_idx = int(total_rows * 0.80)

X_train, y_train = X.iloc[:train_idx], y.iloc[:train_idx]
X_val, y_val     = X.iloc[train_idx:val_idx], y.iloc[train_idx:val_idx]
X_test, y_test   = X.iloc[val_idx:], y.iloc[val_idx:]

print(f"Features: {X.shape[1]} | Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

In [ ]:
# ==============================================================================
# CLASSICAL ML: XGBOOST & RANDOM FOREST (60-MIN)
# ==============================================================================
print("\nTraining Classical ML: XGBoost...")
xgb_results_60m = xgboost_implementation(X_train, y_train, X_val, y_val, X_test, y_test, horizon_name=HORIZON)
save_experiment_results(xgb_results_60m, "XGBoost", HORIZON, y_test)

# Recalculate to enforce m=24
xgb_metrics_60m = calculate_metrics(
    y_true=y_test, y_pred=xgb_results_60m["predictions"], model_name="XGBoost",
    execution_time=xgb_results_60m["metrics"]["Time (s)"], m=MASE_M, y_train=y_train
)
horizon_metrics.append(xgb_metrics_60m)
plot_predictions(y_test, xgb_results_60m["predictions"], "XGBoost", HORIZON, num_steps=PLOT_WINDOW)


print("\nTraining Classical ML: Random Forest...")
rf_results_60m = rf_implementation(X_train, y_train, X_val, y_val, X_test, y_test, horizon_name=HORIZON)
save_experiment_results(rf_results_60m, "Random_Forest", HORIZON, y_test)

# Recalculate to enforce m=24
rf_metrics_60m = calculate_metrics(
    y_true=y_test, y_pred=rf_results_60m["predictions"], model_name="Random_Forest",
    execution_time=rf_results_60m["metrics"]["Time (s)"], m=MASE_M, y_train=y_train
)
horizon_metrics.append(rf_metrics_60m)
plot_predictions(y_test, rf_results_60m["predictions"], "Random_Forest", HORIZON, num_steps=PLOT_WINDOW)

In [ ]:
# ==============================================================================
# DEEP LEARNING: LSTM & VANILLA TRANSFORMER (60-MIN)
# ==============================================================================
print("\nTraining Deep Learning: Power-LSTM...")
lstm_results_60m = run_power_lstm(df=df, target_col=TARGET_COL, seq_length=SEQ_LENGTH, epochs=50, batch_size=32)
save_experiment_results(lstm_results_60m, "LSTM", HORIZON, lstm_results_60m['y_test_real'])
lstm_metrics_60m = calculate_metrics(
    y_true=lstm_results_60m['y_test_real'], y_pred=lstm_results_60m['predictions'],
    model_name="LSTM", execution_time=lstm_results_60m['execution_time'], m=MASE_M, y_train=lstm_results_60m['y_train_real']
)
horizon_metrics.append(lstm_metrics_60m)
plot_predictions(lstm_results_60m['y_test_real'], lstm_results_60m['predictions'], "LSTM", HORIZON, num_steps=PLOT_WINDOW)


print("\nTraining Deep Learning: Vanilla Transformer...")
transformer_results_60m = run_vanilla_transformer(df=df, target_col=TARGET_COL, seq_length=SEQ_LENGTH, epochs=50, batch_size=32)
save_experiment_results(transformer_results_60m, "Vanilla_Transformer", HORIZON, transformer_results_60m['y_test_real'])
transformer_metrics_60m = calculate_metrics(
    y_true=transformer_results_60m['y_test_real'], y_pred=transformer_results_60m['predictions'],
    model_name="Vanilla_Transformer", execution_time=transformer_results_60m['execution_time'], m=MASE_M, y_train=transformer_results_60m['y_train_real']
)
horizon_metrics.append(transformer_metrics_60m)
plot_predictions(transformer_results_60m['y_test_real'], transformer_results_60m['predictions'], "Vanilla_Transformer", HORIZON, num_steps=PLOT_WINDOW)

In [ ]:
# ==============================================================================
# FOUNDATION ML: PATCHTST & CHRONOS-BOLT (60-MIN)
# ==============================================================================
print("\nTraining Foundation ML: PatchTST (RevIN)...")
# Note: For 24 steps, a patch of 8 and stride of 4 works mathematically well!
patch_results_60m = run_keras_patchtst(df=df, target_col=TARGET_COL, seq_length=SEQ_LENGTH, epochs=50, batch_size=32, patch_len=8, stride=4)
save_experiment_results(patch_results_60m, "PatchTST_RevIN", HORIZON, patch_results_60m['y_test_real'])
patch_metrics_60m = calculate_metrics(
    y_true=patch_results_60m['y_test_real'], y_pred=patch_results_60m['predictions'],
    model_name="PatchTST_RevIN", execution_time=patch_results_60m['execution_time'], m=MASE_M, y_train=patch_results_60m['y_train_real']
)
horizon_metrics.append(patch_metrics_60m)
plot_predictions(patch_results_60m['y_test_real'], patch_results_60m['predictions'], "PatchTST_RevIN", HORIZON, num_steps=PLOT_WINDOW)


print("\nTraining Foundation ML: Chronos-Bolt (Univariate Baseline)...")
chronos_results_60m = run_chronos_foundation(df=df, target_col=TARGET_COL, seq_length=SEQ_LENGTH, batch_size=16, model_name="amazon/chronos-bolt-small")
save_experiment_results(chronos_results_60m, "Chronos_Bolt", HORIZON, chronos_results_60m['y_test_real'])
chronos_metrics_60m = calculate_metrics(
    y_true=chronos_results_60m['y_test_real'], y_pred=chronos_results_60m['predictions'],
    model_name="Chronos_Bolt", execution_time=chronos_results_60m['execution_time'], m=MASE_M, y_train=chronos_results_60m['y_train_real']
)
horizon_metrics.append(chronos_metrics_60m)
plot_predictions(chronos_results_60m['y_test_real'], chronos_results_60m['predictions'], "Chronos_Bolt", HORIZON, num_steps=PLOT_WINDOW)


print("\nTraining Foundation ML: Chronos-2 (Multivariate with Covariates)...")
chronos2_results_60m = run_chronos2_multivariate(df=df, target_col=TARGET_COL, prediction_length=SEQ_LENGTH, model_name="amazon/chronos-2", batch_size=8)
save_experiment_results(chronos2_results_60m, "Chronos_2_Multivariate", HORIZON, chronos2_results_60m['y_test_real'])
chronos2_metrics_60m = calculate_metrics(
    y_true=chronos2_results_60m['y_test_real'], y_pred=chronos2_results_60m['predictions'],
    model_name="Chronos_2_Multivariate", execution_time=chronos2_results_60m['execution_time'], m=MASE_M, y_train=chronos2_results_60m['y_train_real']
)
horizon_metrics.append(chronos2_metrics_60m)
plot_predictions(chronos2_results_60m['y_test_real'], chronos2_results_60m['predictions'], "Chronos_2_Multivariate", HORIZON, num_steps=PLOT_WINDOW)

In [ ]:
# ==============================================================================
# FINAL LEADERBOARD GENERATION (60-MINUTE HORIZON)
# ==============================================================================
summary_df_60m = pd.DataFrame(horizon_metrics)

# Sort by R2 Score (Highest to Lowest)
summary_df_60m = summary_df_60m.sort_values(by='R2', ascending=False).reset_index(drop=True)

# Save the final CSV scorecard
summary_df_60m.to_csv(f"../results/{HORIZON}/{HORIZON}_metrics_summary.csv", index=False)

print("\n" + "="*80)
print(f"🏆 {HORIZON.upper()} HORIZON FINAL LEADERBOARD (UNIVARIATE VS MULTIVARIATE) 🏆")
print("="*80)
display(summary_df_60m)